In [1]:
import sys
import numpy as np
from collections import defaultdict, Counter
from seqeval.metrics import classification_report, f1_score, accuracy_score

SEED = 42
np.random.seed(SEED)

DATA_DIR = "data"

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)

python : 3.11.9
numpy  : 2.4.6


In [2]:
def load_data(filepath):
    """Load word-tag data. Returns list of (words, tags) tuples per sentence."""
    sentences = []
    words, tags = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if words:
                    sentences.append((words, tags))
                    words, tags = [], []
            else:
                parts = line.split("\t")
                words.append(parts[0])
                tags.append(parts[1])
    if words:
        sentences.append((words, tags))
    return sentences


train_data = load_data(f"{DATA_DIR}/train_NP.txt")
test_data = load_data(f"{DATA_DIR}/test_NP.txt")

print(f"Training sentences : {len(train_data)}")
print(f"Test sentences     : {len(test_data)}")
print(f"\nExample sentence (train):")
print(f"  Words : {train_data[0][0]}")
print(f"  Tags  : {train_data[0][1]}")

Training sentences : 823
Test sentences     : 77

Example sentence (train):
  Words : ['Rockwell', 'International', 'Corp.', "'s", 'Tulsa', 'unit', 'said', 'it', 'signed', 'a', 'tentative', 'agreement', 'extending', 'its', 'contract', 'with', 'Boeing', 'Co.', 'to', 'provide', 'structural', 'parts', 'for', 'Boeing', "'s", '747', 'jetliners', '.']
  Tags  : ['B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'B-NP', 'I-NP', 'I-NP', 'O']


In [3]:
class HMM:
    """Hidden Markov Model with Add-K smoothing for NP chunking."""

    def __init__(self, k=1.0):
        self.k = k  # smoothing parameter
        self.tags = []  # list of unique tags
        self.vocab = set()  # set of unique words
        self.tag2idx = {}  # tag -> index mapping

        # Raw counts
        self.initial_counts = Counter()  # tag -> count as first tag
        self.transition_counts = defaultdict(Counter)  # tag -> {next_tag: count}
        self.emission_counts = defaultdict(Counter)  # tag -> {word: count}
        self.tag_counts = Counter()  # total count of each tag

        # Log probabilities (computed after training)
        self.log_pi = None  # initial log-probs
        self.log_A = None  # transition log-probs
        self.log_B_cache = {}  # emission log-probs cache

    def train(self, sentences):
        """Estimate HMM parameters from training sentences."""
        # Collect counts
        for words, tags in sentences:
            self.initial_counts[tags[0]] += 1
            for i, (w, t) in enumerate(zip(words, tags)):
                self.vocab.add(w)
                self.tag_counts[t] += 1
                self.emission_counts[t][w] += 1
                if i > 0:
                    self.transition_counts[tags[i - 1]][t] += 1

        self.tags = sorted(self.tag_counts.keys())
        self.tag2idx = {t: i for i, t in enumerate(self.tags)}
        n_tags = len(self.tags)
        V = len(self.vocab)
        total_starts = sum(self.initial_counts.values())

        # --- Initial probabilities (with Add-K smoothing) ---
        self.log_pi = np.zeros(n_tags)
        for i, t in enumerate(self.tags):
            self.log_pi[i] = np.log(
                (self.initial_counts[t] + self.k) / (total_starts + self.k * n_tags)
            )

        # --- Transition probabilities (with Add-K smoothing) ---
        self.log_A = np.zeros((n_tags, n_tags))
        for i, t_prev in enumerate(self.tags):
            total = sum(self.transition_counts[t_prev].values())
            for j, t_next in enumerate(self.tags):
                self.log_A[i, j] = np.log(
                    (self.transition_counts[t_prev][t_next] + self.k)
                    / (total + self.k * n_tags)
                )

        # --- Store vocab size for emission smoothing ---
        self.V = V

        print(f"Tags       : {self.tags}")
        print(f"Vocab size : {V}")
        print(f"Tag counts : {dict(self.tag_counts)}")

    def log_emission(self, tag_idx, word):
        """Compute log P(word | tag) with Add-K smoothing."""
        key = (tag_idx, word)
        if key not in self.log_B_cache:
            tag = self.tags[tag_idx]
            count = self.emission_counts[tag].get(word, 0)
            total = self.tag_counts[tag]
            # Smooth over vocab + 1 (for unknown words)
            self.log_B_cache[key] = np.log(
                (count + self.k) / (total + self.k * (self.V + 1))
            )
        return self.log_B_cache[key]


hmm = HMM(k=1.0)
hmm.train(train_data)

Tags       : ['B-NP', 'I-NP', 'O']
Vocab size : 4570
Tag counts : {'B-NP': 5051, 'I-NP': 5727, 'O': 8394}


In [4]:
print("=" * 50)
print("Initial Probabilities  π(tag):")
print("=" * 50)
for i, t in enumerate(hmm.tags):
    print(f"  π({t:4s}) = {np.exp(hmm.log_pi[i]):.6f}")

print()
print("=" * 50)
print("Transition Probabilities  A(next | prev):")
print("=" * 50)
header = "         " + "  ".join(f"{t:>8s}" for t in hmm.tags)
print(header)
for i, t_prev in enumerate(hmm.tags):
    row = f"{t_prev:4s}  ->  " + "  ".join(
        f"{np.exp(hmm.log_A[i, j]):8.6f}" for j in range(len(hmm.tags))
    )
    print(row)

print()
print("=" * 50)
print("Top-5 Emission Probabilities per Tag:")
print("=" * 50)
for tag in hmm.tags:
    top5 = hmm.emission_counts[tag].most_common(5)
    total = hmm.tag_counts[tag]
    print(f"  {tag}:")
    for word, count in top5:
        prob = (count + hmm.k) / (total + hmm.k * (hmm.V + 1))
        print(f"    {word:20s}  count={count:5d}  P={prob:.6f}")
    print()

Initial Probabilities  π(tag):
  π(B-NP) = 0.656174
  π(I-NP) = 0.001211
  π(O   ) = 0.342615

Transition Probabilities  A(next | prev):
             B-NP      I-NP         O
B-NP  ->  0.029685  0.695428  0.274886
I-NP  ->  0.046011  0.387509  0.566480
O     ->  0.540256  0.000132  0.459613

Top-5 Emission Probabilities per Tag:
  B-NP:
    the                   count=  807  P=0.083974
    a                     count=  365  P=0.038038
    's                    count=  151  P=0.015797
    Mr.                   count=  151  P=0.015797
    The                   count=  125  P=0.013095

  I-NP:
    and                   count=  107  P=0.010487
    million               count=   84  P=0.008254
    %                     count=   83  P=0.008157
    Noriega               count=   67  P=0.006603
    U.S.                  count=   55  P=0.005438

  O:
    ,                     count=  996  P=0.076899
    .                     count=  799  P=0.061705
    of                    count=  459  P=0.035

In [5]:
def forward_algorithm(hmm, words):
    """
    Compute the log-probability of an observation sequence using the Forward Algorithm.

    Parameters
    ----------
    hmm : HMM
        Trained HMM model.
    words : list[str]
        Observed word sequence.

    Returns
    -------
    log_prob : float
        Log-probability of the observation sequence: log P(w1, w2, ..., wT).
    alpha : np.ndarray, shape (T, N)
        Forward trellis (log-space). alpha[t, j] = log α_t(j).
    """
    T = len(words)
    N = len(hmm.tags)
    alpha = np.full((T, N), -np.inf)  # log-space

    # Initialization: α_1(s) = π(s) · B(w_1 | s)
    for j in range(N):
        alpha[0, j] = hmm.log_pi[j] + hmm.log_emission(j, words[0])

    # Recursion: α_t(s) = [Σ_{s'} α_{t-1}(s') · A(s | s')] · B(w_t | s)
    for t in range(1, T):
        for j in range(N):
            # log-sum-exp over all previous states
            log_sum = alpha[t - 1, :] + hmm.log_A[:, j]
            alpha[t, j] = np.logaddexp.reduce(log_sum) + hmm.log_emission(j, words[t])

    # Termination: P(O) = Σ_s α_T(s)
    log_prob = np.logaddexp.reduce(alpha[-1, :])

    return log_prob, alpha

In [6]:
test_log_probs = []

for words, tags in test_data:
    log_prob, _ = forward_algorithm(hmm, words)
    test_log_probs.append(log_prob)

test_log_probs = np.array(test_log_probs)

print(f"Forward Algorithm — Test Sentence Log-Probabilities")
print(f"{'='*60}")
print(f"{'Sentence':>10s}  {'Length':>6s}  {'Log P(sentence)':>20s}")
print(f"{'-'*60}")
for i in range(min(20, len(test_data))):
    words, _ = test_data[i]
    print(f"{i+1:>10d}  {len(words):>6d}  {test_log_probs[i]:>20.4f}")

if len(test_data) > 20:
    print(f"{'...':>10s}")

print(f"{'-'*60}")
print(f"Total test sentences : {len(test_log_probs)}")
print(f"Mean log-probability : {test_log_probs.mean():.4f}")
print(f"Std  log-probability : {test_log_probs.std():.4f}")
print(f"Min  log-probability : {test_log_probs.min():.4f}")
print(f"Max  log-probability : {test_log_probs.max():.4f}")

Forward Algorithm — Test Sentence Log-Probabilities
  Sentence  Length       Log P(sentence)
------------------------------------------------------------
         1      37             -261.8231
         2      27             -187.4174
         3      29             -212.2446
         4      36             -242.4068
         5      31             -217.3535
         6      36             -251.7419
         7      39             -271.7549
         8      25             -177.9311
         9      28             -201.7878
        10      24             -169.6538
        11      16             -114.5829
        12      25             -168.2961
        13      26             -183.0024
        14      47             -334.3213
        15      25             -162.9210
        16      11              -78.6499
        17      32             -217.5961
        18      12              -84.8109
        19      18             -135.8440
        20      26             -158.1175
       ...
---------------

In [7]:
# Pick the first test sentence as an example
sample_words, sample_tags = test_data[0]
sample_log_prob, sample_alpha = forward_algorithm(hmm, sample_words)

print(f"Sample Sentence: {' '.join(sample_words)}")
print(f"Log P(sentence) = {sample_log_prob:.4f}")
print()
print(f"Forward Trellis (log α_t(s)):")
print(f"{'Word':>15s}" + "".join(f"{t:>12s}" for t in hmm.tags))
print("-" * (15 + 12 * len(hmm.tags)))
for t in range(min(len(sample_words), 15)):
    row = f"{sample_words[t]:>15s}"
    for j in range(len(hmm.tags)):
        row += f"{sample_alpha[t, j]:>12.2f}"
    print(row)
if len(sample_words) > 15:
    print(f"{'... (truncated)':>15s}")

Sample Sentence: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
Log P(sentence) = -261.8231

Forward Trellis (log α_t(s)):
           Word        B-NP        I-NP           O
---------------------------------------------------
     Confidence       -9.59      -15.96      -10.54
             in      -20.20      -19.20      -14.12
            the      -17.21      -26.90      -24.35
          pound      -29.88      -26.81      -27.97
             is      -37.51      -36.92      -31.97
         widely      -41.76      -46.39      -42.21
       expected      -51.22      -51.35      -49.70
             to      -59.46      -58.03      -53.57
           take      -63.36      -67.84      -61.61
        another      -69.44      -71.86      -71.75
          sharp      -80.36      -78.30      -79.90
           dive      -89.31     

In [8]:
def viterbi_decode(hmm, words):
    """
    Find the most likely tag sequence using the Viterbi Algorithm.

    Parameters
    ----------
    hmm : HMM
        Trained HMM model.
    words : list[str]
        Observed word sequence.

    Returns
    -------
    best_tags : list[str]
        Most likely tag sequence.
    best_log_prob : float
        Log-probability of the best tag sequence.
    """
    T = len(words)
    N = len(hmm.tags)
    delta = np.full((T, N), -np.inf)  # Viterbi scores (log-space)
    psi = np.zeros((T, N), dtype=int)  # backpointers

    # Initialization
    for j in range(N):
        delta[0, j] = hmm.log_pi[j] + hmm.log_emission(j, words[0])

    # Recursion
    for t in range(1, T):
        for j in range(N):
            scores = delta[t - 1, :] + hmm.log_A[:, j]
            psi[t, j] = np.argmax(scores)
            delta[t, j] = scores[psi[t, j]] + hmm.log_emission(j, words[t])

    # Termination — find the best final state
    best_last = np.argmax(delta[-1, :])
    best_log_prob = delta[-1, best_last]

    # Backtracking
    best_path = [0] * T
    best_path[-1] = best_last
    for t in range(T - 2, -1, -1):
        best_path[t] = psi[t + 1, best_path[t + 1]]

    best_tags = [hmm.tags[idx] for idx in best_path]
    return best_tags, best_log_prob

In [9]:
all_pred_tags = []
all_true_tags = []

for words, true_tags in test_data:
    pred_tags, _ = viterbi_decode(hmm, words)
    all_pred_tags.append(pred_tags)
    all_true_tags.append(true_tags)

print(f"Decoded {len(all_pred_tags)} test sentences.")

Decoded 77 test sentences.


In [10]:
for idx in range(min(5, len(test_data))):
    words, true_tags = test_data[idx]
    pred_tags = all_pred_tags[idx]

    print(f"\n{'='*70}")
    print(f"Sentence {idx + 1}: {' '.join(words)}")
    print(f"{'='*70}")
    print(f"{'Word':>20s}  {'True':>6s}  {'Pred':>6s}  {'Match':>5s}")
    print(f"{'-'*45}")
    for w, tt, pt in zip(words, true_tags, pred_tags):
        match = "✓" if tt == pt else "✗"
        print(f"{w:>20s}  {tt:>6s}  {pt:>6s}  {match:>5s}")


Sentence 1: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
                Word    True    Pred  Match
---------------------------------------------
          Confidence    B-NP    B-NP      ✓
                  in       O       O      ✓
                 the    B-NP    B-NP      ✓
               pound    I-NP    I-NP      ✓
                  is       O       O      ✓
              widely       O       O      ✓
            expected       O       O      ✓
                  to       O       O      ✓
                take       O       O      ✓
             another    B-NP    B-NP      ✓
               sharp    I-NP    I-NP      ✓
                dive    I-NP    I-NP      ✓
                  if       O       O      ✓
               trade    B-NP    B-NP      ✓
             figures    I-NP    I-NP      ✓
                 for 

In [11]:
print("Sequence Labeling Evaluation (seqeval)")
print("=" * 55)
print(classification_report(all_true_tags, all_pred_tags, digits=4))

Sequence Labeling Evaluation (seqeval)
              precision    recall  f1-score   support

          NP     0.6774    0.6632    0.6702       475

   micro avg     0.6774    0.6632    0.6702       475
   macro avg     0.6774    0.6632    0.6702       475
weighted avg     0.6774    0.6632    0.6702       475



In [12]:
entity_f1 = f1_score(all_true_tags, all_pred_tags)
token_acc = accuracy_score(all_true_tags, all_pred_tags)

print(f"Entity-level F1 Score    : {entity_f1:.4f}")
print(f"Token-level Accuracy     : {token_acc:.4f}")

Entity-level F1 Score    : 0.6702
Token-level Accuracy     : 0.8476


In [13]:
# Flatten tags for token-level confusion matrix
flat_true = [t for seq in all_true_tags for t in seq]
flat_pred = [t for seq in all_pred_tags for t in seq]

labels = hmm.tags
cm = np.zeros((len(labels), len(labels)), dtype=int)
label2idx = {l: i for i, l in enumerate(labels)}
for t, p in zip(flat_true, flat_pred):
    cm[label2idx[t], label2idx[p]] += 1

print("Token-Level Confusion Matrix")
print("=" * 45)
print(f"{'':>8s}" + "".join(f"{l:>8s}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l:>8s}" + "".join(f"{cm[i, j]:>8d}" for j in range(len(labels))))

# Per-tag token accuracy
print()
print("Per-Tag Token Accuracy:")
for i, l in enumerate(labels):
    total = cm[i].sum()
    correct = cm[i, i]
    acc = correct / total if total > 0 else 0
    print(f"  {l:>4s}: {correct:>5d} / {total:>5d} = {acc:.4f}")

Token-Level Confusion Matrix
            B-NP    I-NP       O
    B-NP     377      32      66
    I-NP      43     480      56
       O      45      47     750

Per-Tag Token Accuracy:
  B-NP:   377 /   475 = 0.7937
  I-NP:   480 /   579 = 0.8290
     O:   750 /   842 = 0.8907
